# NB5 · Güvenlik bariyerleri ve uyumluluk

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Bu defterde ne yapılıyor

Bu defter modelin başarımını iyileştirmez. Modelin ne zaman konuşmaması gerektiğini
belirler.

Üç bariyer eklenecek. Model kararsız kaldığında susacak. Eğitim popülasyonuna benzemeyen
bir hasta geldiğinde tahmin üretmeyecek. Hakkında yeterli bilgi bulunmayan bir hasta için
karar önermeyecek.

Sonunda bir uyumluluk raporu üretilecek. Rapor ekrana basılacak ve sisteminizin ne
olduğunu, neyi yapmadığını ve hangi soruların açık kaldığını kayda geçirecek.


## Kurulum


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır.')


---

## Önceki defterden gelen kod

Bir önceki defterin sonunda toplanan bloğun tamamını aşağıdaki hücreye yapıştırınız.
İlk satırdaki `#@cdss` işaretini silmeyiniz; o blok bu defterin sonunda yeniden
toplanacak ve bir sonrakine taşınacaktır.

Blok çalıştığında önceki defterlerde yazdığınız her şey yeniden kurulur. İnternetten
veri okuyan satırlar varsa bu hücre birkaç saniye sürebilir.


In [ ]:
#@cdss onceki_defter
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol · Gelen kod


In [ ]:
kit.check_defined('model', 'X_sinama', 'y_sinama', 'olasilik', 'sonuc', 'gerekce_tablosu')


---

## Adım 1 · Bariyerli tahmin

Şu ana kadar sistem her hasta için bir olasılık üretiyordu. Artık üç durumda üretmeyecek.

**Kararsızlık.** Olasılık eşiğin çok yakınındaysa model kararsızdır. Böyle bir durumda
tahmin üretmek yerine kararı klinisyene bırakmak daha dürüsttür.

**Tanımadığı hasta.** Eğitim verisindeki hastalara hiç benzemeyen bir hasta geldiğinde
model yine bir sayı üretir, ama o sayının dayanağı yoktur.

**Eksik bilgi.** Bilgilerinin çoğu boş olan bir hasta için doldurma işlemi bütün
boşlukları ortalama değerlerle kapatır ve model kendinden emin bir sayı üretir. Hakkında
hiçbir şey bilinmeyen bir hasta için sistem karar önerir hâle gelir. Hata mesajı çıkmaz.

Bariyer eşiklerinin ne olacağı klinik bir karardır. İsteme sizin yazdığınız sayılar
bunlardır ve gerekçelerini not etmeniz beklenir.


### İstem 1

```
Sisteme üç güvenlik bariyeri ekleyen bir işlem parçası yaz. Adı guvenli_tahmin olsun;
kendisine bir hastanın bilgilerini alsın.

Sırayla şunlara baksın:
1. Hastanın bilgilerinin yüzde 60'ından fazlası boşsa tahmin üretme. Yetersiz bilgi
   gerekçesiyle kararı klinisyene bırak.
2. Hasta, eğitim grubundaki hastalara hiç benzemiyorsa tahmin üretme. Benzemezliği
   ölçmek için eğitim grubunun tipik değerlerinden ne kadar uzakta olduğuna bak.
3. Bu ikisi geçtiyse olasılığı hesapla. Olasılık karar eşiğine çok yakınsa, yani
   0.50 değerinin 0.05 yakınındaysa kararsız kaldığını bildir.
4. Hiçbiri olmazsa normal kararı ver.

Sonucu bir sözlük olarak geri ver: karar anahtarında metin olarak karar, olasilik
anahtarında sayı ya da None, gerekce anahtarında kısa bir açıklama bulunsun.

Bariyer eşiklerini işlem parçasının dışında, büyük harfli adlarla tanımla ve yanlarına
bunların klinik karar olduğunu yorum olarak yaz.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
guvenli_tahmin adında çalıştırılabilir bir işlem parçası olmalı ve bir sözlük döndürmeli.
Sözlükte karar, olasilik ve gerekce anahtarları bulunmalı.
Bütün bilgileri boş olan bir hasta verildiğinde olasilik değeri None olmalı.
```


In [ ]:
#@cdss bariyerler
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 1


In [ ]:
kit.check_function('guvenli_tahmin')


In [ ]:
import numpy as _np

# Olağan bir hasta
print('Olağan hasta :', guvenli_tahmin(X_sinama[0:1]))

# Bütün bilgileri boş bir hasta
bos = _np.full((1, _np.shape(X_sinama)[1]), _np.nan)
print('Boş hasta    :', guvenli_tahmin(bos))

# Uçlarda bir hasta
uc = _np.asarray(X_sinama[0:1], dtype=float) * 50
print('Uç değerli   :', guvenli_tahmin(uc))


### Python notu · Koşul zinciri ve erken dönüş

Gelen kodda art arda `if` satırları ve her birinin içinde `return` göreceksiniz. Buna
**erken dönüş** denir: Koşul sağlanınca işlem parçası hemen sonucu verir ve alt
satırlara hiç bakmaz.

Bu yapı klinik güvenlik kodunda tercih edilir, çünkü sıra anlam taşır. Önce en ciddi
engel denetlenir, sonra sıradaki. Bilgisi yetersiz bir hasta için eğitim popülasyonuna
benzeyip benzemediğini sormanın anlamı yoktur.

Eşiklerin işlem parçasının dışında tanımlanmasını istememizin sebebi de budur: O sayılar
kodun ortasında saklı kalırsa denetlenemez. Klinik bir sistemde hangi eşikle çalışıldığı
belgelenmesi gereken bir bilgidir.


---

## Adım 2 · Kırmızı takım

Sistemi kendi kurduğunuz senaryolarla zorlayacaksınız. Buna kırmızı takım denir.

Beş girdiden en az biri bariz biçimde bozuk olmamalı, klinik olarak makul ama sıra dışı
bir hasta olmalıdır. Asıl önemli olan o satırdır; yalnızca bariz bozuk girdilerde
başarısız olan bir sistem gerçekte sınanmamıştır.


### İstem 2

```
Sistemi zorlayan beş farklı hasta örneği kur ve her birinde guvenli_tahmin işlem
parçasının ne yaptığını göster.

Beş örnek şöyle olsun:
1. Olağan bir hasta, hiçbir şey değiştirilmemiş.
2. Bir bilgisi fizyolojik olarak imkânsız olan bir hasta.
3. Birçok bilgisi eğitim grubunun tipik değerlerinden çok uzakta olan bir hasta.
4. Bilgilerinin neredeyse tamamı boş olan bir hasta.
5. Bariz bir bozukluğu olmayan ama bir bilgisi eğitim grubundaki en uç değerlere yakın
   olan, klinik olarak makul bir hasta.

Sonuçları bir tablo olarak geri ver: ornek, karar, olasilik ve gerekce sütunları
bulunsun. Tabloyu kirmizi_takim adıyla sakla ve göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
kirmizi_takim adında beş satırlı bir tablo hazır olmalı.
Tabloda ornek, karar, olasilik ve gerekce sütunları bulunmalı.
```


In [ ]:
#@cdss kirmizi_takim
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 2


In [ ]:
kit.check_frame(kirmizi_takim, name='kirmizi_takim',
                required=['ornek', 'karar', 'olasilik', 'gerekce'], min_rows=5)
print()
print(kirmizi_takim.to_string(index=False))


### Tablonun okunması

Beşinci satıra bakınız. Klinik olarak makul ama sıra dışı bir hasta için sistem ne yaptı?
Tahmin ürettiyse, o tahmine güvenilir mi?

Dördüncü satır da önemlidir. Bariyer eklenmeden önce sistem o hasta için kendinden emin
bir olasılık üretiyordu. Bariyeri siz eklediniz; eklemeseydiniz hata mesajı çıkmayacaktı.


---

## Adım 3 · Uyumluluk raporu

Son adımda sisteminizin ne olduğunu kayda geçiren bir rapor üreteceksiniz. Rapor ekrana
basılacak; katılımcılar bunu kendi kurumlarına götürebilir.

Raporun içeriği 16 Eylül dersinde anlatılan çerçeveden gelir: Sistemin amacı, verisi,
ölçülen başarımı, bariyerleri, bilinen sınırlılıkları ve düzenleyici konumu.

Düzenleyici bölüm dikkat ister. Bu alandaki uyum tarihleri Temmuz 2026'da değişmiştir;
Avrupa Birliği'nde tıbbi cihaz içindeki yapay zekâ için geçerli tarih 2 Ağustos 2028'e
ertelenmiştir. Pek çok yapay zekâ aracı hâlâ eski takvimi döndürecek kadar yeni bir
değişikliktir. Aracın verdiği tarihi mutlaka denetleyiniz; bu, kendinden emin icadın
somut bir örneğidir.


### İstem 3

```
Sistemim için bir uyumluluk raporu üreten bir işlem parçası yaz. Adı uyumluluk_raporu
olsun ve raporu tek bir metin olarak geri versin.

Rapor şu başlıkları içersin ve her başlığın altını elimdeki bilgilerle doldur:
  AMAC            -> sistem hangi klinik kararı destekliyor, ne zaman devreye giriyor
  KULLANICI       -> kim kullanacak
  VERI            -> kaç hasta, kaç kayıt, hangi kaynak, tek merkez mi
  BASARIM         -> sonuc sözlüğündeki ölçümler ve ne anlama geldikleri
  BARIYERLER      -> hangi bariyerler var, eşikleri ne, kim belirledi
  SINIRLILIKLAR   -> bu sistem neyi yapmaz, hangi hasta grubunda geçerli değil
  DUZENLEYICI     -> bu yazılım tıbbi cihaz sayılır mı, hangi mevzuata tabi olur,
                     geçerli uyum tarihi nedir
  ACIK_SORULAR    -> bir hukukçunun ve bir klinisyenin denetlemesi gereken noktalar

BASARIM bölümünde sayıları sonuc sözlüğünden çek, elle yazma.

DUZENLEYICI bölümünde her iddianın yanına kendi güven düzeyini yaz. Hiçbirini hukuki
görüş olarak sunma.

Sonra raporu üret, rapor adıyla sakla ve ekrana bas.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
uyumluluk_raporu adında çalıştırılabilir bir işlem parçası olmalı ve bir metin döndürmeli.
rapor adında bir metin hazır olmalı ve yukarıdaki sekiz başlığın tamamını içermeli.
```


In [ ]:
#@cdss uyumluluk
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 3


In [ ]:
kit.check_report(rapor, name='rapor',
                 must_contain=['AMAC', 'KULLANICI', 'VERI', 'BASARIM', 'BARIYERLER',
                               'SINIRLILIKLAR', 'DUZENLEYICI', 'ACIK_SORULAR'])


In [ ]:
print(rapor)


### Raporun denetlenmesi

Üç şeye bakınız.

**DUZENLEYICI bölümündeki tarih.** Avrupa Birliği'nde tıbbi cihaz içindeki yapay zekâ
için geçerli tarih 2 Ağustos 2028'dir. Araç 2027 veya daha eski bir tarih verdiyse eski
takvimi döndürmüş demektir. Düzeltiniz ve bunu not ediniz; aynı davranışı başka
düzenleyici sorularda da bekleyiniz.

**BASARIM bölümündeki sayılar.** Bunlar `sonuc` sözlüğündeki değerlerle aynı mı? Araç
kendi uydurduğu bir sayı yazdıysa rapor güvenilmezdir.

**Varsa atıflar.** Rapordaki her makale ya da kılavuz atfı için DOI ya da resmî belge
numarası isteyiniz. Veremiyorsa atıf kaldırılmalıdır.


---

## Defter sonu · Kodun toplanması

Aşağıdaki hücre önceki defterlerden taşıdığınız kodla bu defterde eklediklerinizi tek
bir blok hâlinde toplar. Çıkan bloğun tamamını kopyalayınız; NB6 defterinin ilk
hücresine yapıştıracaksınız.

Blok ayrıca `cdss_nb5.py` adıyla kaydedilir. Colab oturumu kapandığında bu dosya silinir, bu
nedenle bloğu kendi bilgisayarınızda bir metin dosyasına da kopyalayınız.


In [ ]:
kod = kit.export(save_as='cdss_nb5.py')


## Bu defterde ne yapıldı

Sisteme üç bariyer, bir kırmızı takım denemesi ve bir uyumluluk raporu eklendi.

Bariyerlerin eşikleri sizin belirlediğiniz klinik kararlardır ve raporda yazılıdır. Bir
sistemin hangi eşikle çalıştığı, ne kadar iyi çalıştığı kadar önemli bir bilgidir.

---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
Kullandığınız veri kümesi öğretim için hazırlanmış açık bir kümedir ve kendi kurumunuzun
hasta popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
